# Semiconductor Manufacturing Fault Detection using the SECOM Dataset

This notebook develops a classical machine learning workflow for predicting semiconductor manufacturing pass/fail outcomes from high-dimensional SECOM process parameters.

## Introduction

Semiconductor manufacturing is a complex process with many measured process parameters collected during production. In this notebook, one process run means one row, or one manufacturing observation, in the SECOM dataset. Early detection of faulty process runs can help reduce waste, improve quality control, and support process engineers.

The SECOM dataset is useful for this task because it combines realistic industrial challenges: many features, missing values, noisy measurements, and a rare failure class.

## Problem Formulation

The task is a binary classification problem. For each process run, the model receives a vector of process parameter measurements and predicts whether the run passes or fails quality control.

Let $x_i \in \mathbb{R}^p$ be the process parameter vector for process run $i$, where $p$ is the number of measured parameters. Let $y_i \in \{-1, 1\}$ or equivalently $y_i \in \{0, 1\}$ be the quality label. The goal is to learn a function $f(x_i)$ that estimates the probability or class of failure while generalizing to unseen process runs.

## Dataset Description

The SECOM dataset contains anonymized process parameter measurements from semiconductor manufacturing. The feature matrix includes hundreds of numeric process variables, while the target labels indicate normal and faulty process outcomes.

Because the variables are anonymized, the analysis focuses on statistical behavior, missingness, variance, correlations, and predictive performance rather than domain-specific interpretation of individual process parameters.

## Exploratory Data Analysis

This section starts by importing the required libraries, loading the SECOM data files, and creating one clean dataframe for analysis. After that, the notebook inspects dataset shape, target distribution, missing values, and basic data quality issues.

EDA will be used to identify practical preprocessing needs before model training. Key questions include: How imbalanced is the target? Which process parameters are mostly missing or constant? Are there obvious data quality issues that should influence preprocessing?


In [ ]:
# Imports
from pathlib import Path

# Basic tools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Sklearn modules
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# Evaluation metrics
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    roc_auc_score,
    average_precision_score
)


In [ ]:
# Define paths to the data files
DATA_DIR = Path("../data/raw")

features_path = DATA_DIR / "secom.data"
labels_path = DATA_DIR / "secom_labels.data"

# Load the data
X_raw = pd.read_csv(features_path, sep=r"\s+", header=None)
y_raw = pd.read_csv(labels_path, sep=r"\s+", header=None)

# Check the shapes of the loaded data
X_raw.shape, y_raw.shape

In [ ]:
# Display the first few rows of the features
X_raw.head()

In [ ]:
# Display the first few rows of the labels
y_raw.head()

In [ ]:
# Let's create a clean dataframe by concatenating the features and labels
feature_columns = [f"parameter_{i}" for i in range(X_raw.shape[1])]

X = X_raw.copy()
X.columns = feature_columns

labels = y_raw.copy()
labels.columns = ["target", "timestamp"]

df = X.copy()
df["target"] = labels["target"]
df["timestamp"] = labels["timestamp"]

df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
# Let's check for target class distribution
class_order = [1, -1]
class_labels = ["Fail (1)", "Pass (-1)"]

target_counts = df["target"].value_counts().reindex(class_order, fill_value=0)
target_percentages = df["target"].value_counts(normalize=True).reindex(class_order, fill_value=0) * 100

target_summary = pd.DataFrame({
    "count": target_counts,
    "percentage": target_percentages
})

# Create a bar plot for the target distribution
plt.figure(figsize=(6, 4))

bars = plt.bar(
    class_labels,
    target_summary["count"],
    width=0.25,
    color=["darkorange", "steelblue"]
)

# Add value labels on top of the bars
for i, bar in enumerate(bars):
    height = bar.get_height()
    percentage = target_summary.iloc[i]["percentage"]

    # Add the count label
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        f"{int(height)}",
        ha="center",
        va="bottom"
    )

    # Add the percentage label inside the bar
    plt.text(
        bar.get_x() + bar.get_width() / 2,   # center of the bar on X
        (height / 2) - 10,                   # center of the bar on Y with a small offset
        f"{percentage:.1f}%",                # only the percentage
        ha="center",
        va="center",
        color="white",
        fontsize=12,
        fontweight="bold"
    )

# Set the title and labels for the plot
plt.title("Target Distribution: Pass vs Fail")
plt.xlabel("Target label")
plt.ylabel("Count")

plt.ylim(0, target_summary["count"].max() * 1.1)
plt.show()


The plot shows a clear class imbalance with pass cases dominating the dataset and failure cases appearing much less frequently (*93.4% for the passing class and 6.6% for the failing class*). This means that accuracy alone may be misleading, because a model could mostly predict the majority class and still appear to perform well.

For this reason, later modeling should use imbalance-aware evaluation metrics such as recall, precision, F1-score, confusion matrix, and precision-recall AUC. In this project, correctly identifying failed process runs is especially important, so performance on the minority class will be a key focus.

## Missing Values and Data Quality

SECOM contains missing values across many process parameter measurements, so this section checks both feature-level and row-level data quality. Feature-level missingness asks which process parameters have missing values, while row-level missingness asks whether some process runs have unusually many missing parameter values.

The target-class missingness comparison checks whether pass and fail samples differ in their average percentage of missing parameter values. The goal is to understand how much information is missing, whether missingness is concentrated in specific parameters, and whether some columns are not useful because they are constant.

The analysis below is exploratory. Any preprocessing choices based on missingness or feature quality should later be fit on the training data only, preferably inside a scikit-learn Pipeline, to avoid data leakage.


In [ ]:
# Separate process parameter measurements from the target and timestamp columns
feature_cols = [col for col in df.columns if col not in ["target", "timestamp"]]
X_features = df[feature_cols]

print(f"Number of process runs: {X_features.shape[0]}")
print(f"Number of process parameters: {X_features.shape[1]}")


In [ ]:
# Overall missingness across the feature matrix
total_missing = X_features.isna().sum().sum()
total_values = X_features.size
overall_missing_percentage = total_missing / total_values * 100

missing_overview = pd.DataFrame({
    "total_values": [total_values],
    "missing_values": [total_missing],
    "missing_percentage": [overall_missing_percentage]
})

missing_overview


The dataset has missing values, but the overall percentage alone does not tell the full story. A low global missing percentage can still hide a small group of process parameters with very poor coverage, so the next step is to inspect missingness by feature.


In [ ]:
# Missing values by feature
missing_summary = pd.DataFrame({
    "missing_count": X_features.isna().sum(),
    "missing_percentage": X_features.isna().mean() * 100
})

missing_summary = missing_summary.sort_values("missing_percentage", ascending=False)

missing_summary.head(20)


In [ ]:
# Count how many features exceed useful missingness thresholds
missing_thresholds = [0, 5, 10, 20, 40, 60, 80]

missing_threshold_summary = pd.DataFrame({
    "missing_threshold_percentage": missing_thresholds,
    "number_of_features_above_threshold": [
        (missing_summary["missing_percentage"] > threshold).sum()
        for threshold in missing_thresholds
    ]
})

missing_threshold_summary


In [ ]:
# Visualize the features with the highest missing value percentages
top_missing = missing_summary.head(20).sort_values("missing_percentage")

plt.figure(figsize=(8, 6))
bars = plt.barh(
    top_missing.index,
    top_missing["missing_percentage"],
    color="steelblue"
)

for bar in bars:
    width = bar.get_width()
    plt.text(
        width + 1,
        bar.get_y() + bar.get_height() / 2,
        f"{width:.1f}%",
        va="center"
    )

plt.title("Top 20 Features by Missing Value Percentage")
plt.xlabel("Missing values (%)")
plt.ylabel("Feature")
plt.xlim(0, 100)
plt.tight_layout()
plt.show()


The missing values are not evenly distributed. In this dataset, many process parameters have at least one missing value, but the most important issue is the group of parameters with very high missingness. Several columns are missing for more than 40% of the process runs, and a few are missing for more than 80%.

This suggests two preprocessing actions to consider later: remove features with extremely high missingness and impute the remaining missing values. Both actions should be learned from the training split only.


In [ ]:
# Missing values by process run / row
row_missing_summary = pd.DataFrame({
    "missing_count": X_features.isna().sum(axis=1),
    "missing_percentage": X_features.isna().mean(axis=1) * 100,
    "target": df["target"]
})

row_missing_summary[["missing_count", "missing_percentage"]].describe()


In [ ]:
# Distribution of missing values per process run
plt.figure(figsize=(7, 4))

plt.hist(
    row_missing_summary["missing_percentage"],
    bins=30,
    color="steelblue",
    edgecolor="black"
)

plt.axvline(
    row_missing_summary["missing_percentage"].mean(),
    color="darkorange",
    linestyle="--",
    label="Mean"
)

plt.axvline(
    row_missing_summary["missing_percentage"].median(),
    color="black",
    linestyle=":",
    label="Median"
)

plt.title("Distribution of Missing Values per Process Run")
plt.xlabel("Missing values per row (%)")
plt.ylabel("Number of process runs")
plt.legend()
plt.tight_layout()
plt.show()


The row-level missingness distribution shows that most process runs have a relatively low percentage of missing parameter values, concentrated around 3–6%. The mean is slightly higher than the median, which indicates a right-skewed distribution caused by a small number of process runs with unusually high missingness. A few observations have more than 10% missing values, with extreme cases above 20%.

In [ ]:
# Compare the average percentage of missing parameter values by target class
target_order = [1, -1]
target_labels = ["Fail (1)", "Pass (-1)"]

row_missing_by_target = (
    row_missing_summary
    .groupby("target")[["missing_count", "missing_percentage"]]
    .mean()
    .reindex(target_order)
)

row_missing_by_target.index = target_labels
row_missing_by_target


In [ ]:
# Plot average missingness by target class
plt.figure(figsize=(6, 4))

bars = plt.bar(
    row_missing_by_target.index,
    row_missing_by_target["missing_percentage"],
    width=0.35,
    color=["darkorange", "steelblue"]
)

for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        f"{height:.2f}%",
        ha="center",
        va="bottom"
    )

plt.title("Average Missing Values by Target Class")
plt.xlabel("Target class")
plt.ylabel("Average missing values per row (%)")
plt.ylim(0, row_missing_by_target["missing_percentage"].max() * 1.2)
plt.tight_layout()
plt.show()


The row-level view checks whether missing data is concentrated in a small number of process runs, where each process run is one row in the dataset. Comparing target-class missingness helps determine whether pass and fail samples have noticeably different average percentages of missing parameter values.

In this dataset, the average missingness by class is similar, so missingness appears to be more of a general data quality issue than an obvious direct separator between pass and fail outcomes.


In [ ]:
# Detect constant features, which do not help classification
unique_counts = X_features.nunique(dropna=True)
constant_features = unique_counts[unique_counts <= 1]

print(f"Number of constant features: {len(constant_features)}")

constant_features.head(20)

In [ ]:
# Compact data quality summary for preprocessing decisions
data_quality_summary = pd.DataFrame({
    "metric": [
        "features_with_any_missing_values",
        "features_with_more_than_40_percent_missing",
        "features_with_more_than_80_percent_missing",
        "constant_features"
    ],
    "value": [
        (missing_summary["missing_percentage"] > 0).sum(),
        (missing_summary["missing_percentage"] > 40).sum(),
        (missing_summary["missing_percentage"] > 80).sum(),
        len(constant_features)
    ]
})

data_quality_summary


This data quality check motivates the preprocessing strategy for the next section. The dataset should not be passed directly into a model without handling missing values and uninformative columns.

A realistic first preprocessing plan is to remove columns with very high missingness, remove constant features, impute the remaining missing values, and scale features when the model requires it. These steps should be implemented in a leakage-safe way using the training data only.


## Preprocessing Strategy

The SECOM dataset contains hundreds of anonymized numerical process parameters, missing values, constant or near-constant features, and a strongly imbalanced binary target. Because of this, preprocessing is a critical part of the modeling workflow.

The first preprocessing step is to define the feature matrix and target vector, then create a stratified train/test split. Stratification is important because the failure class is rare, so both splits should preserve the pass/fail class ratio as much as possible.

Later model pipelines will handle learned preprocessing steps such as imputation, constant-feature removal, and scaling. These steps must be fit only on the training split or inside cross-validation folds. The held-out test set should be used only for final evaluation.


The split below creates the objects that all later modeling code should use. No imputer, scaler, feature filter, PCA step, or model should be fitted before this split.


In [ ]:
# Define features and target
X = df[feature_cols]
y = df["target"]

# Create a stratified train/test split before fitting any preprocessing step
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

X_train.shape, X_test.shape, y_train.value_counts(normalize=True), y_test.value_counts(normalize=True)


## Baseline Models

The first baseline model will be Logistic Regression because it is simple, fast, and easy to interpret as a starting point. Since Logistic Regression is sensitive to feature scale and cannot handle missing values directly, it is wrapped in a `scikit-learn` Pipeline.

The pipeline below applies median imputation, removes constant features, scales the remaining parameters, and then fits a class-weighted Logistic Regression model. Defining the pipeline before fitting helps ensure that preprocessing is learned only from the training data.


In [ ]:
# Baseline Logistic Regression pipeline

# Sklearn tries to balance the classess universally proportionally
log_reg_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("variance_filter", VarianceThreshold(threshold=0.0)),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced", #  use balanced class weights to handle class imbalance
        random_state=42 
    ))
])

log_reg_pipeline.fit(X_train, y_train)

y_pred = log_reg_pipeline.predict(X_test)
y_proba = log_reg_pipeline.predict_proba(X_test)[:, 1]

In [ ]:
# Evaluate the baseline model
cm = confusion_matrix(y_test, y_pred, labels=[-1, 1])

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Fail (-1)", "Pass (1)"]
)

disp.plot(cmap="Blues")
plt.title("Confusion Matrix - Logistic Regression Baseline")
plt.show()

In [ ]:
print(classification_report(
    y_test,
    y_pred,
    labels=[1, -1],
    target_names=["Fail (1)", "Pass (-1)"]
))

In [ ]:
fail_class_index = list(log_reg_pipeline.named_steps["classifier"].classes_).index(1)
y_fail_proba = log_reg_pipeline.predict_proba(X_test)[:, fail_class_index]

roc_auc = roc_auc_score(y_test == 1, y_fail_proba)
pr_auc = average_precision_score(y_test == 1, y_fail_proba)

print(f"ROC-AUC: {roc_auc:.3f}")
print(f"PR-AUC: {pr_auc:.3f}")

## Handling Class Imbalance

Faulty process runs are expected to be much less frequent than normal process runs. Accuracy alone can therefore be misleading because a model may appear strong while missing most failures.

This section will compare strategies such as class weights, threshold adjustment, and metrics focused on the minority class.

## Model Evaluation

Model evaluation will use a held-out test set and cross-validation on the training set. Metrics will include confusion matrix, precision, recall, F1-score, ROC-AUC, and precision-recall AUC.

For this fault detection task, recall for the failure class is especially important, but it must be balanced against false alarms.

## PCA / Dimensionality Reduction

Because the dataset has hundreds of features, PCA may help reduce noise and create lower-dimensional representations for some models.

PCA will be evaluated inside a Pipeline after imputation and scaling, ensuring that the transformation is learned only from training data during cross-validation.

## Discussion and Limitations

This section will interpret model results, compare trade-offs between detecting failures and avoiding false alarms, and describe limitations of the dataset and modeling approach.

Important limitations may include anonymized features, small number of failure examples, missing values, and limited ability to infer causal process explanations.

## Conclusion

The conclusion will summarize the best-performing approach, the main lessons from the analysis, and practical recommendations for future work.

The final project should demonstrate a complete and leakage-safe classical ML workflow suitable for course review.